# Build the Modeling Table — Phases 0-2

Implements **Build Sequence Phases 0, 1, and 2** from the project spec
(`README.md` §3, §9):

| Phase | Task | Output |
|---|---|---|
| **0** | Sort all 59 `cps_filers_clean.csv` columns into piles (§3.4) | (assignment; `data_dictionary.md` written at end of Phase 1) |
| **1** | Filter to filer units (`depstat==0`, §3.2), confirm `eff_rate` target, **reconstruct tax-unit income**, build share features | clean filer table + `data_dictionary.md` |
| **2** | **Freeze** the modeling table (80/20, fixed seed) — *blocks all downstream work* | `train.csv`, `test.csv` |

**Decisions carried from the spec (with the recommended defaults):**
- Filer unit = **(A) `depstat == 0`** (non-dependent filers) — §3.2, recommended.
- Target = **`eff_rate`** used directly (already `100 × fedtaxac / adjginc`) — §3.3.
- Negative effective rates (refundable-credit filers) are **kept, not clipped** — §3.3.
- Feature scope = **core only** — income level + income-composition shares +
  filing/marital/household/age/geography. Optional context axes and all
  sensitive axes (race, ethnicity, immigration, sex, …) are **excluded** from
  `X`, per the locked income / filing-status / geography framing (§3.4, §10.4).
- Split = **80/20, `random_state=42`** — §3.5.

**Two amendments to the spec, both forced by the data (see Phase 1 and `data_dictionary.md`):**

1. **§3.2 option (A) alone is not sufficient.** The filter selects one row per return, but the
   income columns on that row are one *person's*. The target is scoped to the whole return, so
   unit income is reconstructed (dependents via `depstat`, the absent spouse via the family
   residual in `ftotval`) before any feature is derived. Median `adjginc / inctot` was **1.77**
   for joint filers vs **1.00** for single; after reconstruction the spread across filing
   statuses falls from **0.77 to 0.11** and correlation with `adjginc` rises **0.815 → 0.983**.
2. **§3.4's `filestat` glossary is wrong.** The IPUMS codebook shipped in this repo has
   `1/2/3` as *joint, both <65 / one 65+ / both 65+* — all the same filing status, split by age —
   not `joint / joint-sep / sep`. There is no married-filing-separately category in this data.
   Collapsed to `filing_status` (joint / head_of_household / single) so the twin comparison can
   flip filing status without silently flipping age bracket too.


## Setup

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RANDOM_STATE = 42   # fixed seed for every stochastic step (reproducibility, §4.1)
SHARE_FLOOR  = 1000 # min UNIT income for a well-defined income share; below this the
                    # denominator is too small and shares explode -> set them to 0

# Per the IPUMS codebook shipped at data/raw/data_dictionary.csv, filestat is:
#   1=Joint both <65 | 2=Joint one <65 one 65+ | 3=Joint both 65+ | 4=Head of household | 5=Single
# Codes 1/2/3 are the SAME filing status split by age -- not joint/separate as
# README §3.4 glosses them. The data agrees: filestat in (1,2,3) is 100% marst==1,
# and filestat==3 has a minimum age of 65. Collapsed to filing_status below so the
# age split lives in `age` (already a feature) instead of hiding inside filing status.
JOINT_FILESTAT    = (1, 2, 3)
FILING_STATUS_MAP = {1: 1, 2: 1, 3: 1, 4: 4, 5: 5}   # -> 1=joint, 4=head_of_household, 5=single

INCOME_COMPONENTS = ["incwage", "incbus", "incint", "incdivid", "incretir", "incss", "incrent"]

# Resolve the repo root whether this notebook runs from notebooks/ or the root.
ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

RAW_CSV   = ROOT / "data" / "raw" / "cps_filers_clean.csv"
LABELS    = ROOT / "data" / "raw" / "data_dictionary.csv"   # variable labels + value codes
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(RAW_CSV)
labels = pd.read_csv(LABELS).set_index("variable")["label"].to_dict()

print("Loaded:", RAW_CSV.relative_to(ROOT))
print("Shape :", df_raw.shape, "(spec expects 64,696 x 59)")
assert df_raw.shape == (64696, 59), "input file does not match the spec's 64,696 x 59"


Loaded: data/raw/cps_filers_clean.csv
Shape : (64696, 59) (spec expects 64,696 x 59)


## Phase 0 — Column pile-sort (leakage firewall)

Every one of the 59 columns is assigned to **exactly one** pile. Misassignment
causes data leakage — a model that peeks at the answer (§3.4, §10.1).
**When timing is ambiguous, quarantine.**

Pile vocabulary:

| Pile | Meaning |
|---|---|
| `target` | The value we predict (`eff_rate`). |
| `feature` | Pre-tax, known before any tax is computed → enters `X`, directly or via a derived column. |
| `optional_feature` | Legitimate pre-tax axis, but **out of scope** for the locked framing → documented, not in `X`. |
| `quarantine` | Target components or post-tax/credit outputs → **never** a feature (leakage). |
| `weight` | Survey weights — used for weighted validation, **never** in `X`. |
| `sensitive` | Race / ethnicity / immigration / sex / disability — excluded from the equity framing (§10.4). |
| `drop` | Identifiers, administrative fields, redundant/household-level columns, and structural keys. |

The pile assignment is made here; `data_dictionary.md` is **written at the end of
Phase 1**, once the derived columns it also documents exist.


In [2]:
# Authoritative pile assignment for all 59 columns: col -> (pile, rationale).
# This dict is the single source of truth; data_dictionary.md is generated from it.
COLUMN_PILES = {
    # --- target -------------------------------------------------------------
    "eff_rate":   ("target", "Prediction target: 100 x fedtaxac / adjginc (percent), scoped to the tax RETURN."),

    # --- features (pre-tax; core framing) -----------------------------------
    "inctot":     ("feature", "Pre-tax total personal income; the filer's own contribution to unit_inctot."),
    "ftotval":    ("feature", "Pre-tax total FAMILY income. Sole recoverable source of the absent spouse's "
                              "income (see Phase 1 step 4); enters X only via unit_inctot / the shares."),
    "incwage":    ("feature", "Source of wage_share (income composition; core equity signal)."),
    "incbus":     ("feature", "Source of business_share; can be negative (losses)."),
    "incint":     ("feature", "Source of interest_share."),
    "incdivid":   ("feature", "Source of dividend_share; preferentially taxed capital income."),
    "incretir":   ("feature", "Source of retirement_share; nulls imputed to 0."),
    "incss":      ("feature", "Source of socsec_share; partially/untaxed."),
    "incrent":    ("feature", "Source of rent_share; can be negative (losses)."),
    "filestat":   ("feature", "Tax filer status. Codes 1/2/3 are all JOINT (split by age, per codebook) -> "
                              "collapsed to filing_status. CPS-tax-model assigned; flagged (§3.4)."),
    "marst":      ("feature", "Marital status; structural driver independent of imputed filestat."),
    "nchild":     ("feature", "Number of own children; drives dependent credits/deductions."),
    "nchlt5":     ("feature", "Young children (<5); CTC / childcare relevance."),
    "famsize":    ("feature", "Family size; household scale."),
    "age":        ("feature", "Age; retirement / senior treatment."),
    "statefip":   ("feature", "State (FIPS); geographic axis. Categorical (51 levels) -> one-hot at model time."),

    # --- optional features (out of scope for the locked framing) ------------
    "educ":       ("optional_feature", "Education; context axis, not in the core income/filing/geography framing."),
    "empstat":    ("optional_feature", "Employment status; optional context axis."),
    "wkswork1":   ("optional_feature", "Weeks worked last year; optional context axis."),
    "uhrsworkly": ("optional_feature", "Usual hours/week; optional context axis."),
    "classwkr":   ("optional_feature", "Class of worker (self-employed vs wage); optional context axis."),
    "pension":    ("optional_feature", "Pension coverage; optional context axis."),
    "firmsize":   ("optional_feature", "Firm size; optional context axis."),

    # --- quarantine (target components / post-tax / credits) ----------------
    "fedtaxac":   ("quarantine", "Numerator of the target (federal tax after credits)."),
    "adjginc":    ("quarantine", "Denominator of the target (AGI). Used ONLY to validate feature scope, never in X."),
    "spmfedtaxac":("quarantine", "SPM-unit federal tax; same outcome, different grouping. Leakage."),
    "eitcred":    ("quarantine", "Refundable credit; computed as part of the tax outcome."),
    "ctccrd":     ("quarantine", "Child Tax Credit; post-tax credit."),
    "actccrd":    ("quarantine", "Additional (refundable) CTC; post-tax credit."),
    "margtax":    ("quarantine", "Marginal tax rate; direct function of the tax computation."),
    "taxinc":     ("quarantine", "Taxable income; post-deduction, downstream of AGI."),
    "fica":       ("quarantine", "Payroll tax; a computed tax outcome (kept out to keep 'pre-tax' clean)."),

    # --- survey weights (validation only) -----------------------------------
    "asecwt":     ("weight", "ASEC person weight; population-representative validation. Never in X."),
    "asecwth":    ("weight", "ASEC household weight; validation weight. Never in X."),

    # --- sensitive (excluded from the equity framing, §10.4) ----------------
    "sex":        ("sensitive", "Sensitive; include only if the equity framing explicitly covers it (it does not)."),
    "race":       ("sensitive", "Race; not an agreed equity axis. Never a causal driver in this project."),
    "hispan":     ("sensitive", "Hispanic origin; not an agreed equity axis."),
    "citizen":    ("sensitive", "Citizenship; immigration axis, out of framing."),
    "nativity":   ("sensitive", "Nativity; immigration axis, out of framing."),
    "yrimmig":    ("sensitive", "Year of immigration; immigration axis, out of framing."),
    "vetstat":    ("sensitive", "Veteran status; sensitive, out of framing."),
    "diffany":    ("sensitive", "Disability; sensitive, out of framing."),

    # --- drop (ids, admin, redundant, structural keys) ----------------------
    "year":       ("drop", "Constant survey year; identifier/admin."),
    "serial":     ("drop", "Household serial number; identifier. Structural key for tax-unit assembly, not a feature."),
    "month":      ("drop", "Survey month; admin."),
    "cpsid":      ("drop", "CPSID household record; identifier."),
    "cpsidp":     ("drop", "CPSID person record; identifier."),
    "cpsidv":     ("drop", "Validated longitudinal identifier; identifier."),
    "asecflag":   ("drop", "ASEC flag; admin."),
    "pernum":     ("drop", "Person number; identifier. Target of the depstat pointer, not a feature."),
    "ftype":      ("drop", "Family type; administrative grouping."),
    "spmfamunit": ("drop", "SPM unit id; grouping identifier. NOT used to key the family "
                           "(an SPM unit can span several families -- 4,173 do)."),
    "spmwt":      ("drop", "SPM unit weight; not used (filer-level analysis)."),
    "spmftotval": ("drop", "SPM unit cash income; wrong grouping for the tax unit (ftotval is used instead)."),
    "hhincome":   ("drop", "Total household income; household spans multiple tax units, too coarse."),
    "occ2010":    ("drop", "Occupation (479 levels); not in the §3.4 feature set."),
    "ind":        ("drop", "Industry; not in the §3.4 feature set."),
    "depstat":    ("drop", "Dependency-status pointer: selects filer units (==0) and assigns dependents to "
                           "their filer. Structural, not a feature; constant (0) after filtering."),
}

VALID_PILES = {"target", "feature", "optional_feature", "quarantine", "weight", "sensitive", "drop"}

# Integrity checks: every column classified exactly once, nothing invented.
assert set(COLUMN_PILES) == set(df_raw.columns), (
    "mismatch: missing=%s extra=%s"
    % (set(df_raw.columns) - set(COLUMN_PILES), set(COLUMN_PILES) - set(df_raw.columns))
)
assert len(COLUMN_PILES) == 59, len(COLUMN_PILES)
assert all(p in VALID_PILES for p, _ in COLUMN_PILES.values())

pile_of = {c: p for c, (p, _) in COLUMN_PILES.items()}
counts = pd.Series(pile_of).value_counts()
print("All 59 columns classified exactly once. Pile counts:")
print(counts.to_string())


All 59 columns classified exactly once. Pile counts:
feature             16
drop                16
quarantine           9
sensitive            8
optional_feature     7
weight               2
target               1


## Phase 1 — Filer-unit filter, target confirmation & feature engineering

1. Filter to **filer units** — `depstat == 0` (non-dependent filers, §3.2 option A).
2. **Confirm the target** — `eff_rate == 100 x fedtaxac / adjginc`, always defined (`adjginc > 0`).
3. **Impute** `incretir` nulls to 0 (§3.4).
4. **Reconstruct the tax unit** — *amends §3.2/§3.4.* The target is scoped to the whole
   tax **return**, but every income column on a filer row is that one person's own. Rebuild
   income at return scope before deriving anything from it.
5. Build **unit-scoped income-composition shares** (guarded to 0 when unit income < $1,000)
   plus `spouse_income_share`, and collapse `filestat` → `filing_status`.
6. Assemble the modeling table `X + eff_rate + asecwt` and run a **leakage assertion**.

> **Why step 4 exists.** `adjginc` (the target's denominator) and `fedtaxac` (its numerator)
> are imputed by Census at the **filing** level, so for a joint return they cover both spouses.
> `inctot` and the seven income components are **person**-level. Median `adjginc / inctot` was
> **1.77** for joint filers versus **1.00** for single filers — meaning the model was being asked
> to predict a rate whose denominator it could not see, for ~40% of rows. Left unfixed this
> corrupts both headline deliverables: SHAP would charge the missing spouse's income to
> `filing_status`/`marst`, and the twin comparison's single → joint flip would report a gap that
> is mostly "married people often have a second earner" rather than anything the tax code does.


In [3]:
# --- 1. Filter to filer units (depstat == 0) --------------------------------
df = df_raw[df_raw["depstat"] == 0].copy()
print(f"Filer units (depstat==0): {len(df):,} rows  (spec expects 61,231)")
assert len(df) == 61231

# --- 2. Confirm the target --------------------------------------------------
assert (df["adjginc"] > 0).all(), "adjginc must be > 0 for eff_rate to be defined"
recomputed = 100 * df["fedtaxac"] / df["adjginc"]
max_diff = (df["eff_rate"] - recomputed).abs().max()
print(f"Target check: max |eff_rate - 100*fedtaxac/adjginc| = {max_diff:.2e}  (float noise)")
assert max_diff < 1e-6, "eff_rate does not match 100*fedtaxac/adjginc"

n_neg = int((df["fedtaxac"] < 0).sum())
print(f"eff_rate range: [{df['eff_rate'].min():.2f}, {df['eff_rate'].max():.2f}]  "
      f"(percent)")
print(f"Negative-rate filers (fedtaxac<0, refundable credits): {n_neg:,} "
      f"({100*n_neg/len(df):.1f}%) — KEPT, not clipped (§3.3)")

# --- 3. Impute incretir nulls to 0 ------------------------------------------
n_null_retir = int(df["incretir"].isna().sum())
df["incretir"] = df["incretir"].fillna(0)
print(f"Imputed incretir: {n_null_retir:,} nulls -> 0")


Filer units (depstat==0): 61,231 rows  (spec expects 61,231)
Target check: max |eff_rate - 100*fedtaxac/adjginc| = 1.42e-14  (float noise)
eff_rate range: [-49.99, 33.37]  (percent)
Negative-rate filers (fedtaxac<0, refundable credits): 7,344 (12.0%) — KEPT, not clipped (§3.3)
Imputed incretir: 42,778 nulls -> 0


In [4]:
# --- 4. Reconstruct income at TAX-UNIT scope --------------------------------
# The tax unit = the filer + everyone on their return. Two pieces are missing from
# the filer's own row, and they are recovered differently:
#
#   dependents — present in the extract as their own rows. `depstat` holds the
#                pernum of the filer claiming them, within the same `serial`.
#   spouse     — NOT present at all. This extract keeps one spouse per couple
#                (26,871 persons have marst==1 "spouse present", and 26,355
#                households hold exactly one of them), so the spouse's income
#                cannot be read off any row. It survives only inside `ftotval`,
#                the pre-tax family total, which still counts them. So the
#                family residual  ftotval - sum(inctot of persons present)  IS
#                the absent spouse.
#
# Evidence the residual is the spouse and not noise: families containing no joint
# filer show a residual of $0 at the 25th, 50th AND 75th percentile, while families
# containing one show a median of ~$50,000.
#
# `ftotval` is a FAMILY total and an SPM unit can span several families (4,173 do),
# so the residual is keyed on the family, not on spmfamunit. Keying it on
# spmfamunit instead produces 2,978 impossible negative residuals; the family key
# produces 148 (rounding/topcoding against inctot), which are clipped to 0.
persons = df_raw.copy()
persons[INCOME_COMPONENTS] = persons[INCOME_COMPONENTS].fillna(0)
persons["_fam"] = list(zip(persons["serial"], persons["ftotval"]))  # ftotval is constant within a family

# (a) dependents -> the filer who claims them
deps = persons[persons["depstat"] != 0]
dep_totals = deps.groupby(["serial", "depstat"])[["inctot", *INCOME_COMPONENTS]].sum()
dep_totals.index.names = ["serial", "pernum"]
filer_key = df.set_index(["serial", "pernum"]).index
for col in ["inctot", *INCOME_COMPONENTS]:
    df["dep_" + col] = np.nan_to_num(filer_key.map(dep_totals[col]).to_numpy().astype(float))

# (b) family residual -> the joint filer of that family (split if somehow >1)
by_fam = persons.groupby("_fam")
residual = (by_fam["ftotval"].first() - by_fam["inctot"].sum()).clip(lower=0)
df["_fam"] = list(zip(df["serial"], df["ftotval"]))
is_joint = df["filestat"].isin(JOINT_FILESTAT)
n_joint_in_fam = df[is_joint].groupby("_fam").size()
df["spouse_inc"] = np.where(
    is_joint,
    df["_fam"].map(residual).fillna(0.0) / df["_fam"].map(n_joint_in_fam).fillna(1.0),
    0.0,
)

df["unit_inctot"] = df["inctot"] + df["dep_inctot"] + df["spouse_inc"]
for col in INCOME_COMPONENTS:
    df["unit_" + col] = df[col] + df["dep_" + col]

print(f"Dependents assigned to a filer: {len(deps):,} rows carrying "
      f"${deps['inctot'].clip(lower=0).sum():,.0f} of income")
print(f"Joint filers given a spouse: {int(is_joint.sum()):,}  "
      f"({100 * (df.loc[is_joint, 'spouse_inc'] == 0).mean():.1f}% get $0 — one-earner couples)")

# --- Scope validation against adjginc (quarantined; used as a ruler, never in X) --
scope = pd.DataFrame({
    "before  adjginc/inctot":      (df["adjginc"] / df["inctot"].replace(0, np.nan)),
    "after   adjginc/unit_inctot": (df["adjginc"] / df["unit_inctot"].replace(0, np.nan)),
}).groupby(df["filestat"]).median()
scope.index = [f"{k} ({'joint' if k in JOINT_FILESTAT else 'head' if k == 4 else 'single'})"
               for k in scope.index]
print("\nFeature scope vs. target scope — median ratio by filestat (1.00 = aligned):")
print(scope.round(2).to_string())
print(f"\n  spread across statuses: {scope.iloc[:, 0].max() - scope.iloc[:, 0].min():.2f}"
      f"  ->  {scope.iloc[:, 1].max() - scope.iloc[:, 1].min():.2f}")
print(f"  correlation with adjginc: {df['inctot'].corr(df['adjginc']):.4f}"
      f"  ->  {df['unit_inctot'].corr(df['adjginc']):.4f}")
print("  (filestat 3 lands below 1.00 because for both-65+ couples much of Social")
print("   Security income is excluded from AGI — real, not a scope error.)")

assert (df["unit_inctot"] >= df["inctot"]).all(), "reconstruction lost income"
assert df["spouse_inc"].ge(0).all() and df.loc[~is_joint, "spouse_inc"].eq(0).all()


Dependents assigned to a filer: 3,465 rows carrying $58,193,077 of income
Joint filers given a spouse: 26,871  (6.2% get $0 — one-earner couples)

Feature scope vs. target scope — median ratio by filestat (1.00 = aligned):
            before  adjginc/inctot  after   adjginc/unit_inctot
1 (joint)                     1.77                         1.00
2 (joint)                     1.58                         0.97
3 (joint)                     1.28                         0.88
4 (head)                      1.00                         1.00
5 (single)                    1.00                         1.00

  spread across statuses: 0.77  ->  0.12
  correlation with adjginc: 0.8154  ->  0.9817
  (filestat 3 lands below 1.00 because for both-65+ couples much of Social
   Security income is excluded from AGI — real, not a scope error.)


In [5]:
# --- 5. Unit-scoped income-composition shares -------------------------------
# Numerator = filer + dependents for that source; denominator = unit_inctot.
# Guarded to 0 when unit_inctot < SHARE_FLOOR ($1000): below that the denominator
# is too small and shares explode (inctot=$50 with incint=$500 -> 1000%), which
# would produce nonsense splits and nonsense SHAP attributions. Shares stay
# negative for genuine business/rent losses above the floor (kept, §3.4).
#
# The spouse's income is recoverable only as a TOTAL — ftotval carries no
# breakdown by source — so it cannot be distributed across the seven shares.
# It gets its own column instead. That keeps the shares additive and makes the
# unobserved portion legible to SHAP as its own bar, rather than silently
# deflating the other seven for every joint filer.
SHARE_SRC = {
    "wage_share":       "incwage",
    "business_share":   "incbus",
    "interest_share":   "incint",
    "dividend_share":   "incdivid",
    "retirement_share": "incretir",
    "socsec_share":     "incss",
    "rent_share":       "incrent",
}

base_ok = df["unit_inctot"] >= SHARE_FLOOR
n_guarded = int((~base_ok).sum())
for share, src in SHARE_SRC.items():
    df[share] = np.where(base_ok, df["unit_" + src] / df["unit_inctot"], 0.0)
df["spouse_income_share"] = np.where(base_ok, df["spouse_inc"] / df["unit_inctot"], 0.0)

SHARE_COLS = [*SHARE_SRC, "spouse_income_share"]
print(f"Built {len(SHARE_COLS)} unit-scoped shares; guarded (set to 0) where "
      f"unit_inctot < {SHARE_FLOOR}: {n_guarded:,} rows")
print(df[SHARE_COLS].agg(["min", "median", "max"]).T.round(3).to_string())
n_gt1 = int((df[SHARE_COLS].abs() > 1).any(axis=1).sum())
print(f"Rows with any |share| > 1 after floor: {n_gt1} ({100 * n_gt1 / len(df):.2f}%) — "
      f"genuine offsetting business/rent losses, kept.")

# --- 6. Collapse filestat -> filing_status ----------------------------------
# 1/2/3 are all joint returns (codebook), differing only by whether each spouse is
# 65+. Keeping the raw code would hand the RF an age bracket disguised as a filing
# status — redundant with `age`, and it would make the twin flip incoherent
# ("5 -> 3" would mean "become married AND become 65+", an unattributable gap).
df["filing_status"] = df["filestat"].map(FILING_STATUS_MAP)
assert df["filing_status"].notna().all(), "unmapped filestat code"
print("\nfilestat -> filing_status  (1=joint, 4=head_of_household, 5=single)")
print(pd.crosstab(df["filestat"], df["filing_status"]).to_string())

# --- 7. Assemble the modeling table (core features + target + weight) --------
NUMERIC_FEATURES = ["unit_inctot", *SHARE_COLS, "age", "nchild", "nchlt5", "famsize"]
CATEGORICAL_FEATURES = ["filing_status", "marst", "statefip"]  # codes; encode at model time
FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "eff_rate"
WEIGHT = "asecwt"

model_df = df[FEATURE_COLS + [TARGET, WEIGHT]].copy()

# --- Leakage assertion ------------------------------------------------------
# Engineered columns are not in COLUMN_PILES, so the firewall is enforced on the
# raw columns each one is built from: every source must sit in the `feature` pile.
DERIVED_FROM = {
    "unit_inctot":         ["inctot", "ftotval"],
    "spouse_income_share": ["inctot", "ftotval"],
    "filing_status":       ["filestat"],
    **{share: [src, "inctot", "ftotval"] for share, src in SHARE_SRC.items()},
}
FORBIDDEN = {"quarantine", "sensitive", "drop", "target", "weight"}
for col in FEATURE_COLS:
    sources = DERIVED_FROM.get(col, [col])
    bad = [s for s in sources if pile_of.get(s) in FORBIDDEN or s not in pile_of]
    assert not bad, f"LEAKAGE: feature {col!r} is derived from {bad}"

# raw amounts must NOT survive as columns — only the derived shares
assert not (set(INCOME_COMPONENTS) & set(model_df.columns)), "raw income components leaked into table"
assert not ({"inctot", "ftotval", "filestat"} & set(model_df.columns)), "pre-reconstruction column leaked"
assert not (set(model_df.columns) & {"adjginc", "fedtaxac"}), "target component leaked into table"
assert pile_of[TARGET] == "target" and pile_of[WEIGHT] == "weight"
assert model_df[FEATURE_COLS].isna().sum().sum() == 0, "unexpected nulls in features"
assert model_df[TARGET].isna().sum() == 0
assert np.isfinite(model_df[NUMERIC_FEATURES].to_numpy()).all(), "non-finite value in X"

print("\nLeakage assertion passed. Modeling table:", model_df.shape)
print("Features:", FEATURE_COLS)
print("Nulls in table:", int(model_df.isna().sum().sum()))


Built 8 unit-scoped shares; guarded (set to 0) where unit_inctot < 1000: 126 rows
                       min  median    max
wage_share           0.000   0.689  1.555
business_share      -2.515   0.000  1.000
interest_share       0.000   0.000  1.180
dividend_share       0.000   0.000  1.000
retirement_share     0.000   0.000  1.000
socsec_share         0.000   0.000  1.000
rent_share          -0.345   0.000  1.000
spouse_income_share  0.000   0.000  1.554
Rows with any |share| > 1 after floor: 68 (0.11%) — genuine offsetting business/rent losses, kept.

filestat -> filing_status  (1=joint, 4=head_of_household, 5=single)
filing_status      1     4      5
filestat                         
1              20219     0      0
2               1979     0      0
3               4673     0      0
4                  0  5583      0
5                  0     0  28777

Leakage assertion passed. Modeling table: (61231, 18)
Features: ['unit_inctot', 'wage_share', 'business_share', 'interest_share', 'di

In [6]:
# Generate data_dictionary.md from COLUMN_PILES (Phase 0 deliverable).
ORDER = ["target", "feature", "optional_feature", "quarantine", "weight", "sensitive", "drop"]
PILE_DESC = {
    "target":           "The value we predict.",
    "feature":          "Pre-tax; enters the model matrix X (directly or via a derived column).",
    "optional_feature": "Legitimate pre-tax axis, out of scope for the locked framing; documented, not in X.",
    "quarantine":       "Target components or post-tax/credit outputs; never a feature (leakage).",
    "weight":           "Survey weight; weighted validation only, never in X.",
    "sensitive":        "Sensitive axis excluded from the equity framing (§10.4).",
    "drop":             "Identifier, admin, redundant/household-level, or a structural key.",
}

lines = []
lines.append("# Data Dictionary & Column Pile-Sort")
lines.append("")
lines.append("**Source:** `data/raw/cps_filers_clean.csv` — 64,696 rows x 59 columns "
             "(IPUMS CPS ASEC 2024, income year 2023), one row per person.")
lines.append("")
lines.append("> Auto-generated by `notebooks/build_modeling_table.ipynb` (Phase 0). "
             "Do not edit by hand — edit `COLUMN_PILES` in the notebook and re-run.")
lines.append("")
lines.append("This is the **leakage firewall** required by the spec (README §3.4, §9 Phase 0): "
             "every column is assigned to exactly one pile. When column timing is ambiguous, it is quarantined.")
lines.append("")

# summary table
lines.append("## Pile summary")
lines.append("")
lines.append("| Pile | Count | Meaning |")
lines.append("|---|---:|---|")
for p in ORDER:
    lines.append(f"| `{p}` | {int(counts.get(p, 0))} | {PILE_DESC[p]} |")
lines.append(f"| **total** | **{len(COLUMN_PILES)}** | |")
lines.append("")

# per-pile detail tables (preserve dict order within each pile)
for p in ORDER:
    cols = [c for c in COLUMN_PILES if pile_of[c] == p]
    if not cols:
        continue
    lines.append(f"## `{p}` ({len(cols)})")
    lines.append("")
    lines.append("| Column | Label | Rationale |")
    lines.append("|---|---|---|")
    for c in cols:
        lines.append(f"| `{c}` | {labels.get(c, '')} | {COLUMN_PILES[c][1]} |")
    lines.append("")

# --- tax-unit reconstruction (Phase 1 step 4) -------------------------------
lines.append("## Unit of analysis: the tax RETURN, not the person")
lines.append("")
lines.append("The target is scoped to the whole filing unit — `fedtaxac` and `adjginc` are imputed by "
             "Census at the **filing** level — while every income column on a filer row is that one "
             "**person's** own. Before reconstruction, the median `adjginc / inctot` was "
             "**1.77** for joint filers against **1.00** for single filers: the model would have been "
             "asked to predict a rate whose denominator it could not see, for ~40% of rows.")
lines.append("")
lines.append("Income is therefore rebuilt at return scope before any feature is derived from it:")
lines.append("")
lines.append("| Piece | How it is recovered |")
lines.append("|---|---|")
lines.append("| Filer | Their own row. |")
lines.append("| Dependents | Present as rows; `depstat` points at the claiming filer's `pernum` within `serial`. |")
lines.append("| Spouse | **Absent from the extract entirely** — one spouse is kept per couple. Recovered as the "
             "family residual `ftotval - sum(inctot of persons present)`. |")
lines.append("")
lines.append("The residual is the spouse, not noise: families with **no** joint filer show a residual of "
             "$0 at the 25th, 50th and 75th percentile; families with one show a median of **~$50,000**. "
             "It is keyed on the **family** (`ftotval` is a family total), not on `spmfamunit` — an SPM unit "
             "can span several families, and 4,173 do. Keying on `spmfamunit` yields 2,978 impossible "
             "negative residuals versus 148 on the family key (clipped to 0).")
lines.append("")
lines.append("| Scope check (median ratio, 1.00 = aligned) | joint | head | single | spread |")
lines.append("|---|---:|---:|---:|---:|")
lines.append(f"| before — `adjginc / inctot` | {scope.iloc[0, 0]:.2f} | {scope.iloc[3, 0]:.2f} "
             f"| {scope.iloc[4, 0]:.2f} | {scope.iloc[:, 0].max() - scope.iloc[:, 0].min():.2f} |")
lines.append(f"| after — `adjginc / unit_inctot` | {scope.iloc[0, 1]:.2f} | {scope.iloc[3, 1]:.2f} "
             f"| {scope.iloc[4, 1]:.2f} | {scope.iloc[:, 1].max() - scope.iloc[:, 1].min():.2f} |")
lines.append("")
lines.append(f"Correlation with `adjginc`: **{df['inctot'].corr(df['adjginc']):.3f} -> "
             f"{df['unit_inctot'].corr(df['adjginc']):.3f}**. `adjginc` is quarantined and is used here "
             "purely as a ruler to check scope — it never enters `X`. A better-scoped *pre-tax income* "
             "feature is not leakage: income level is the primary legitimate driver of an effective rate, "
             "and knowing the denominator of a ratio does not reveal the ratio.")
lines.append("")

# --- engineered feature set -------------------------------------------------
lines.append("## Modeling feature set `X` (built in Phase 1)")
lines.append("")
lines.append("Only `feature`-pile columns are used, and every engineered column is checked back to the "
             "pile of each raw column it is built from. Raw income amounts are transformed into "
             "**income-composition shares** (the core equity signal); the raw amounts do not enter `X`.")
lines.append("")
lines.append(f"Each share is `(filer + dependents for that source) / unit_inctot`, set to **0 when "
             f"`unit_inctot < {SHARE_FLOOR}`**: below that floor the denominator is too small and shares "
             "explode (e.g. `inctot=$50, incint=$500` -> 1000%), producing nonsense splits and nonsense "
             "SHAP attributions. Shares stay negative for genuine business/rent losses above the floor "
             "(kept, §3.4).")
lines.append("")
lines.append("The spouse's income is recoverable only as a **total** — `ftotval` carries no breakdown by "
             "source — so it cannot be distributed across the seven shares. It gets its own column, "
             "`spouse_income_share`, which keeps the shares additive and makes the unobserved portion "
             "legible to SHAP as its own bar instead of silently deflating the other seven for every "
             "joint filer.")
lines.append("")
lines.append("| Feature | Type | Derivation |")
lines.append("|---|---|---|")
lines.append("| `unit_inctot` | numeric | `inctot + dependents' inctot + spouse residual` — pre-tax income of the whole return. |")
for share, src in SHARE_SRC.items():
    lines.append(f"| `{share}` | numeric | `(unit {src}) / unit_inctot`, 0 when `unit_inctot < {SHARE_FLOOR}`. |")
lines.append(f"| `spouse_income_share` | numeric | `spouse residual / unit_inctot` — the portion of unit "
             f"income with no source breakdown; 0 for non-joint filers. |")
lines.append("| `filing_status` | categorical | `filestat` collapsed: **1=joint (raw 1/2/3), 4=head_of_household, 5=single**. |")
lines.append("| `marst` | categorical | Marital status. |")
lines.append("| `statefip` | categorical | State FIPS (51 levels) -> one-hot at model time. |")
lines.append("| `nchild` | numeric | Number of own children. |")
lines.append("| `nchlt5` | numeric | Own children under 5. |")
lines.append("| `famsize` | numeric | Family size. |")
lines.append("| `age` | numeric | Age. |")
lines.append("")
lines.append("**Target:** `eff_rate` (percent). **Retained non-feature column:** `asecwt` (weighted validation, §3.5).")
lines.append("")
lines.append("### Why `filestat` was collapsed")
lines.append("")
lines.append("README §3.4 glosses the codes as `1=joint, 2=joint/sep, 3=sep`. The IPUMS codebook shipped "
             "at `data/raw/data_dictionary.csv` says otherwise:")
lines.append("")
lines.append("| Code | Codebook | In this file |")
lines.append("|---|---|---|")
lines.append("| 1 | Joint, both < 65 | 100% `marst`=1, ages 17–64 |")
lines.append("| 2 | Joint, one < 65, one 65+ | 100% `marst`=1, median age 65 |")
lines.append("| 3 | Joint, both 65+ | 100% `marst`=1, **minimum age 65** |")
lines.append("| 4 | Head of household | mixed `marst` |")
lines.append("| 5 | Single | mixed `marst` |")
lines.append("")
lines.append("Codes 1/2/3 are the **same filing status split by age**, and there is no "
             "married-filing-separately category in this data at all. Keeping the raw code would hand the "
             "model an age bracket disguised as a filing status — redundant with `age`, which is already a "
             "feature — and would make the twin comparison incoherent: flipping `5 -> 3` means *become "
             "married **and** become 65+*, a gap no single attribute could own.")
lines.append("")
lines.append("## Filer-unit filter, target & negative rates")
lines.append("")
lines.append(f"- **Filer unit (§3.2, option A + reconstruction):** keep `depstat == 0` (non-dependent filers) "
             f"-> {len(df):,} rows, each rebuilt to whole-return income scope.")
lines.append("- **Target (§3.3):** `eff_rate = 100 x fedtaxac / adjginc`; all rows have `adjginc > 0`, so it is always defined.")
lines.append("- **Negative rates (§3.3):** ~12% of filer rows have `fedtaxac < 0` (refundable credits exceed tax owed). "
             "These negative effective rates are **kept, not clipped** — the net-progressive tail is a finding.")
lines.append("")

DICT_MD = ROOT / "data_dictionary.md"
DICT_MD.write_text("\n".join(lines) + "\n")
print("Wrote", DICT_MD.relative_to(ROOT), f"({len(lines)} lines)")


Wrote data_dictionary.md (185 lines)


In [7]:
# --- Persist the clean filer / modeling table (Phase 1 output) --------------
MODEL_CSV = PROCESSED / "cps_filers_modeling.csv"
MODEL_PARQUET = PROCESSED / "cps_filers_modeling.parquet"
model_df.to_csv(MODEL_CSV, index=False)
model_df.to_parquet(MODEL_PARQUET, index=False)
print("Wrote", MODEL_CSV.relative_to(ROOT), model_df.shape)
print("Wrote", MODEL_PARQUET.relative_to(ROOT))
model_df.head()


Wrote data/processed/cps_filers_modeling.csv (61231, 18)
Wrote data/processed/cps_filers_modeling.parquet


,unit_inctot,wage_share,business_share,interest_share,dividend_share,retirement_share,socsec_share,rent_share,spouse_income_share,age,nchild,nchlt5,famsize,filing_status,marst,statefip,eff_rate,asecwt
0,50838.0,0.000000,0.0,0.000039,0.000000,0.000000,0.212420,0.0,0.787541,53,0,0,2,1,1,23,4.063351,1559.99
1,35987.0,0.000000,0.0,0.207714,0.005002,0.333454,0.453831,0.0,0.000000,72,0,0,1,5,6,23,2.532584,1521.46
2,71435.0,0.000000,0.0,0.001400,0.000000,0.000000,0.000000,0.0,0.998600,62,1,0,3,1,1,23,5.104318,1500.23
3,54000.0,0.000000,0.0,0.000000,0.000000,0.000000,0.666667,0.0,0.000000,58,0,0,2,1,1,23,0.000000,1351.18
4,80007.0,0.537453,0.0,0.000075,0.000000,0.000000,0.000000,0.0,0.462472,57,1,0,3,1,1,23,6.465511,1351.18


## Phase 2 — Freeze the modeling table (blocking dependency)

Split the modeling table **80/20 with `random_state=42`** into `train.csv` /
`test.csv`. Per §3.5 this freeze is the **only blocking dependency** — once
frozen, no column may be added, and Phases 3-7 (RF, SHAP, twin, validation,
demo) run against these exact files. A `freeze_manifest.json` records the seed,
shapes, and content hashes so any later drift is detectable.


In [8]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    model_df, test_size=0.20, random_state=RANDOM_STATE, shuffle=True,
)
print(f"Split 80/20 (seed={RANDOM_STATE}):  train {train_df.shape}  test {test_df.shape}")
assert len(train_df) + len(test_df) == len(model_df)
assert set(train_df.index).isdisjoint(test_df.index), "train/test overlap!"

# --- Freeze verification: the split must be representative -------------------
def profile(name, d):
    neg = 100 * (d["eff_rate"] < 0).mean()
    return dict(split=name, rows=len(d),
                eff_mean=round(d["eff_rate"].mean(), 3),
                eff_std=round(d["eff_rate"].std(), 3),
                pct_negative=round(neg, 2),
                unit_inctot_median=round(d["unit_inctot"].median(), 1),
                pct_joint=round(100 * d["filing_status"].eq(1).mean(), 2))

profile_tbl = pd.DataFrame([profile("full", model_df),
                            profile("train", train_df),
                            profile("test", test_df)]).set_index("split")
print("\nRepresentativeness (train/test should track full):")
print(profile_tbl.to_string())


Split 80/20 (seed=42):  train (48984, 18)  test (12247, 18)

Representativeness (train/test should track full):
        rows  eff_mean  eff_std  pct_negative  unit_inctot_median  pct_joint
split                                                                       
full   61231     5.352   10.216         11.99             67400.0      43.88
train  48984     5.310   10.286         12.21             67331.5      43.91
test   12247     5.521    9.929         11.15             67520.0      43.80


In [9]:
import hashlib, json

TRAIN_CSV = PROCESSED / "train.csv"
TEST_CSV  = PROCESSED / "test.csv"
train_df.to_csv(TRAIN_CSV, index=False)
test_df.to_csv(TEST_CSV, index=False)
# parquet copies for fast, type-stable downstream loads
train_df.to_parquet(PROCESSED / "train.parquet", index=False)
test_df.to_parquet(PROCESSED / "test.parquet", index=False)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

manifest = {
    "source": "data/processed/cps_filers_modeling.csv",
    "filer_unit_filter": "depstat == 0",
    "unit_reconstruction": {
        "why": "eff_rate is scoped to the tax RETURN (fedtaxac/adjginc are imputed at the "
               "filing level) but every income column is person-level. Median adjginc/inctot "
               "was 1.77 for joint filers vs 1.00 for single.",
        "unit_income": "inctot + dependents' inctot + spouse residual",
        "dependents_via": "depstat pointer -> filer pernum within serial",
        "spouse_via": "family residual: ftotval - sum(inctot of persons present), clipped at 0",
        "family_key": ["serial", "ftotval"],
        "joint_filestat_codes": list(JOINT_FILESTAT),
        "scope_check_median_adjginc_ratio": {
            "before_by_filestat": {int(k): round(v, 3) for k, v in
                                   (df["adjginc"] / df["inctot"].replace(0, np.nan))
                                   .groupby(df["filestat"]).median().items()},
            "after_by_filestat": {int(k): round(v, 3) for k, v in
                                  (df["adjginc"] / df["unit_inctot"].replace(0, np.nan))
                                  .groupby(df["filestat"]).median().items()},
        },
        "corr_with_adjginc": {"before": round(df["inctot"].corr(df["adjginc"]), 4),
                              "after":  round(df["unit_inctot"].corr(df["adjginc"]), 4)},
    },
    "filing_status_map": {str(k): v for k, v in FILING_STATUS_MAP.items()},
    "filing_status_labels": {"1": "joint", "4": "head_of_household", "5": "single"},
    "share_floor": SHARE_FLOOR,
    "share_basis": "unit_inctot",
    "random_state": RANDOM_STATE,
    "test_size": 0.20,
    "shuffle": True,
    "target": TARGET,
    "weight_retained": WEIGHT,
    "feature_cols": FEATURE_COLS,
    "n_features": len(FEATURE_COLS),
    "rows": {"full": len(model_df), "train": len(train_df), "test": len(test_df)},
    "sha256": {"train.csv": sha256(TRAIN_CSV), "test.csv": sha256(TEST_CSV)},
    "note": "FROZEN modeling table (README Phase 2). Do not add columns; "
            "downstream phases 3-7 read these exact files.",
}
MANIFEST = PROCESSED / "freeze_manifest.json"
MANIFEST.write_text(json.dumps(manifest, indent=2) + "\n")

print("FROZEN:")
for p in (TRAIN_CSV, TEST_CSV, MANIFEST):
    print("  ", p.relative_to(ROOT))
print("\ntrain.csv sha256:", manifest["sha256"]["train.csv"][:16], "...")
print("test.csv  sha256:", manifest["sha256"]["test.csv"][:16], "...")


FROZEN:
   data/processed/train.csv
   data/processed/test.csv
   data/processed/freeze_manifest.json

train.csv sha256: 3f56170b3514db95 ...
test.csv  sha256: 286577a80698787d ...


## Definition of Done — Phases 0-2

- [x] **Phase 0** — every one of the 59 columns assigned to exactly one pile,
      documented in `data_dictionary.md` (generated from `COLUMN_PILES`).
- [x] **Phase 1** — filer-unit table built (`depstat==0`, 61,231 rows);
      `eff_rate` target confirmed; **income reconstructed to tax-return scope**;
      unit-scoped composition shares engineered; `filestat` collapsed to
      `filing_status`; leakage assertion passed.
- [x] **Phase 2** — modeling table **frozen** 80/20 (`random_state=42`) into
      `train.csv` / `test.csv`, with `freeze_manifest.json` (seed, shapes, hashes,
      and the full reconstruction record).

**Artifacts produced**

| Path | Phase |
|---|---|
| `data_dictionary.md` | 0 (written at end of 1) |
| `data/processed/cps_filers_modeling.{csv,parquet}` | 1 |
| `data/processed/train.{csv,parquet}`, `test.{csv,parquet}` | 2 |
| `data/processed/freeze_manifest.json` | 2 |

**Downstream (out of scope here):** Phase 3 Random Forest, Phase 4 SHAP,
Phase 5 twin comparison, Phase 6 IRS SOI validation, Phase 7 Streamlit demo.

Notes for whoever picks up Phase 3+:

- Categorical features (`filing_status`, `marst`, `statefip`) are kept as codes and
  should be one-hot encoded at model time.
- `filing_status` is **1=joint, 4=head_of_household, 5=single**. The twin
  comparison's headline flip is `5 -> 1` (single → joint), holding `age` and every
  income column constant.
- `spouse_income_share` is the portion of unit income with no source breakdown. It is
  0 for every non-joint filer, so read its SHAP bar as *"a second earner is on this
  return"*, not as an income type.
- The 80/20 split is **row-random, not group-aware**: 8,917 families contain 2+
  separate filers, so related returns can straddle train and test. Expect test R² to
  be mildly optimistic and say so when reporting metrics (§4.2).
